# 03. Face Analysis — A Pipeline of Several Models

> Instead of one big model doing everything, we **chain several small models** and split the work.
> Find faces → cut out just the face → feed it to the age/gender, emotion, and head-pose models.

### What you will learn
1. Matching each model's own **input size · output names** (they differ per model card)
2. Using the first model's result (boxes) to build the second model's input (cropped faces)
3. Reading the outputs: regression (age), class probabilities (gender · emotion), angles (head pose)
4. Why this fits the NPU well — small models with a fixed input size

### The whole flow
```
photo ─▶ [find faces 300x300] ─▶ boxes ─crop (20% margin)─▶ face photos
                                          ├─▶ [age·gender 62x62] ─▶ age, female/male prob.
                                          ├─▶ [emotion 64x64]    ─▶ 5 probabilities
                                          └─▶ [head pose 60x60]  ─▶ yaw · pitch · roll angles
```

| Model | Input | Output |
|---|---|---|
| face-detection-retail-0005 | 300 x 300 BGR | `[1,1,N,7]` (notebook 01) |
| age-gender-recognition-retail-0013 | 62 x 62 BGR | `age_conv3` (age ÷ 100), `prob` ([female, male]) |
| emotions-recognition-retail-0003 | 64 x 64 BGR | 5 probabilities: neutral · happy · sad · surprise · angry |
| head-pose-estimation-adas-0001 | 60 x 60 BGR | `angle_y_fc` · `angle_p_fc` · `angle_r_fc` (degrees) |

## 1. Setup — load the four models
You can pass a file path straight to `compile_model` (it runs `read_model` for you inside).
For a model with several outputs, you pick each one by its **output name** — so we print the names first.

In [ ]:
import os
import time
from pathlib import Path

import cv2
import numpy as np
import openvino as ov
from PIL import Image
from IPython.display import display

ROOT = Path(os.environ.get("EDGE_ROOT", "../..")).resolve()       # program folder (lab.bat tells us); this notebook is in notebooks/en/
MODELS = Path(os.environ.get("EDGE_MODELS", ROOT / "models"))      # model folder
core = ov.Core()
print("Model folder:", MODELS)
print("Devices:", core.available_devices)

def pick(*prefer):
    """Pick the first device in the list that this PC has."""
    for d in prefer:
        if d in core.available_devices:
            return d
    return "CPU"

DEVICE = pick("NPU", "GPU", "CPU")                  # small fixed-size models — a good fit for the NPU
F = MODELS / "face"
det = core.compile_model(F / "face-detection-retail-0005.xml", DEVICE)
agegen = core.compile_model(F / "age-gender-recognition-retail-0013.xml", DEVICE)
emo = core.compile_model(F / "emotions-recognition-retail-0003.xml", DEVICE)
pose = core.compile_model(F / "head-pose-estimation-adas-0001.xml", DEVICE)
print("Device:", DEVICE)
for name, c in [("face", det), ("age·gender", agegen), ("emotion", emo), ("head pose", pose)]:
    print(f"  {name:10s} input {c.input(0).shape}  outputs {[o.any_name for o in c.outputs]}")

## 2. Photo

In [ ]:
PHOTO = "sample.jpg"          # photo to use when there is no webcam (in the same folder as this notebook)

def show(bgr, width=720):
    """Show an OpenCV photo (BGR) inside the notebook."""
    img = Image.fromarray(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB))
    img.thumbnail((width, width))
    display(img)

def grab(index=0):
    """Take one frame from the webcam. If there is no webcam, use the PHOTO file."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    frame = None
    for _ in range(5):                   # some webcams give dark first frames
        ok, f = cap.read()
        if ok:
            frame = f
    cap.release()
    if frame is None:
        frame = cv2.imread(PHOTO)
        assert frame is not None, f"No webcam and no {PHOTO} — put a photo file name in PHOTO"
        print("No webcam, so using the file:", PHOTO)
    return frame

frame = grab()
print("Photo size (height, width, channels):", frame.shape)
show(frame)

## 3. Find faces → crop them
Same as notebook 01. One difference: we crop the box **a little larger (margin)**. The later models were trained on face photos that show the hair and chin too.
If you crop too tightly, the age and emotion results go wrong.

In [ ]:
def blob(bgr, size):
    """Shared preprocessing for the models: resize to a square, keep BGR · 0~255 · NCHW."""
    return cv2.resize(bgr, (size, size)).transpose(2, 0, 1)[None].astype(np.float32)

def find_faces(bgr, conf=0.5, margin=0.2):
    h, w = bgr.shape[:2]
    dets = det(blob(bgr, 300))[det.output(0)].reshape(-1, 7)
    out = []
    for image_id, _, score, x1, y1, x2, y2 in dets:
        if image_id < 0:
            break
        if score < conf:
            continue
        x1, y1, x2, y2 = x1 * w, y1 * h, x2 * w, y2 * h
        mx, my = (x2 - x1) * margin, (y2 - y1) * margin        # extra room: margin × the box size
        box = (int(max(0, x1 - mx)), int(max(0, y1 - my)), int(min(w, x2 + mx)), int(min(h, y2 + my)))
        out.append((box, float(score)))
    return out

found = find_faces(frame)
print("Found", len(found), "faces")

In [ ]:
# The face each model really 'sees' — resized to its input, then enlarged again to show (smaller = blurrier)
for (x1, y1, x2, y2), s in found[:3]:
    face = frame[y1:y2, x1:x2]
    views = [cv2.resize(cv2.resize(face, (n, n)), (160, 160), interpolation=cv2.INTER_NEAREST) for n in (62, 64, 60)]
    print(f"face {face.shape[1]}x{face.shape[0]} → age·gender 62 · emotion 64 · head pose 60")
    show(np.hstack(views), 480)

## 4. Three models for each cropped face

**Reading the outputs**
- **Age** — one number (regression). The model learned age divided by 100, so we **× 100**.
- **Gender** — two probabilities `[female, male]` (they add up to 1). We pick the bigger one.
- **Emotion** — five probabilities. The biggest one (`argmax`) is the answer, and the probabilities themselves mean something too ("happy 0.55, neutral 0.40" means it is unsure).
- **Head pose** — three angles (degrees).

```
      up/down (pitch)          left/right (yaw)        tilt (roll)
          ↑                    ←  ○  →                   ╱○
          ○  nodding           shaking head "no"        tilting head
          ↓
```

In [ ]:
EMO = ["neutral", "happy", "sad", "surprise", "angry"]

def analyze(face):
    r = agegen(blob(face, 62))
    age = float(r[agegen.output("age_conv3")].squeeze()) * 100       # pick the output by its name
    female, male = r[agegen.output("prob")].squeeze()

    p = emo(blob(face, 64))[emo.output(0)].squeeze()                 # five probabilities

    r = pose(blob(face, 60))
    yaw = float(r[pose.output("angle_y_fc")].squeeze())      # left/right (degrees)
    pitch = float(r[pose.output("angle_p_fc")].squeeze())    # up/down
    roll = float(r[pose.output("angle_r_fc")].squeeze())     # tilt
    return {"age": round(age), "gender": "female" if female > male else "male", "gender_conf": round(float(max(female, male)), 2),
            "emotion": EMO[int(p.argmax())], "emotion_probs": np.round(p, 2),
            "yaw": round(yaw), "pitch": round(pitch), "roll": round(roll)}

view = frame.copy()
results = []
for i, ((x1, y1, x2, y2), s) in enumerate(found):
    info = analyze(frame[y1:y2, x1:x2])
    results.append(info)
    print(f"face {i}:", info)
    cv2.rectangle(view, (x1, y1), (x2, y2), (0, 200, 0), 2)
    cv2.putText(view, f"#{i} age {info['age']} yaw {info['yaw']}", (x1, max(20, y1 - 8)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 200, 0), 2)
show(view)

`cv2.putText` can only draw basic English letters and numbers — so on the photo we write short English text, and print the full result as text. (To draw other scripts such as Korean on a photo, use PIL's `ImageDraw.text` with a font that has those letters.)

### Emotion probabilities as bars

In [ ]:
import matplotlib.pyplot as plt     # chart — we write the labels in English (non-Latin text like Korean needs extra font setup)

if results:
    p = results[0]["emotion_probs"]
    plt.figure(figsize=(5, 2.4))
    plt.bar(["neutral", "happy", "sad", "surprise", "angry"], p, color="#1f5f7a")
    plt.ylim(0, 1); plt.title("face 0 emotion"); plt.show()

## 5. Head pose in words — make a rule
Angles are numbers, so they are hard to use as they are. We choose a **threshold** and turn them into "front / left / right / up / down".
Find the threshold (15 degrees here) by trying it yourself. It changes with the camera position and the person.

In [ ]:
def facing(yaw, pitch, th=15):
    lr = "left" if yaw > th else ("right" if yaw < -th else "")
    ud = "up" if pitch < -th else ("down" if pitch > th else "")
    return (lr + " " + ud).strip() or "front"

for i, info in enumerate(results):
    print(f"face {i}: yaw {info['yaw']:+d}° · pitch {info['pitch']:+d}° → {facing(info['yaw'], info['pitch'])}")

The **sign** of the angle (is left + or −?) can look reversed if the camera image is mirrored. Turn your head and check it yourself.

## 6. How many ms per face? — the cost of a pipeline
When you chain models, their times add up too. With 5 faces, the three analysis models run 5 times.

In [ ]:
if found:
    (x1, y1, x2, y2), _ = found[0]
    face = frame[y1:y2, x1:x2]
    for name, fn in [("find", lambda: find_faces(frame)), ("analyze x3", lambda: analyze(face))]:
        t0 = time.perf_counter()
        for _ in range(20):
            fn()
        print(f"{name:10s} {(time.perf_counter() - t0) / 20 * 1000:6.2f} ms  ({DEVICE})")

## Common errors
| Symptom | Cause · Fix |
|---|---|
| `RuntimeError ... output ... not found` | The output name is different → use the names printed in section 1 |
| Strange age (between 0 and 1) | You forgot the × 100 |
| Error when a face touches the photo edge | When cropping, keep inside the photo with `max(0, …)` · `min(w, …)` |
| Emotion is always 'neutral' | The face is too small or cropped too tightly → `margin`, move closer |

## Summary
- Pipeline = the key part is the code that turns the **previous model's output into the next model's input** (crop · resize).
- Outputs are read in different ways: regression (a number) · classification (probabilities) · angles. Probabilities also tell you "how sure" the model is.
- Several small fixed-size models → a good fit for the NPU.

## Try it
1. Change `margin` to 0 and to 0.5. Do the age and emotion change?
2. Change `DEVICE` and compare the times in section 6.
3. Take several webcam shots (`frame = grab()`) with different faces, and see how the emotion probabilities move.
4. Make a rule like "alert when you look at the front for more than 3 seconds".
   <details><summary>Hint</summary>

   Keep the webcam open, loop, and measure how long `facing(...) == "front"` stays true. When it breaks, reset the time to 0.
   </details>

---
# 🛠 Mini project — Focus Meter

**What to build**: Watch the webcam for 20 seconds. For each frame, check "is there a face, and is it looking to the front?". Give a **focus score (front ratio %)**
and show the **distribution of expressions** during that time. (A tiny version of an online-class or study-timer tool.)

**What to use**: `find_faces` · `analyze` · `facing` (this notebook), `live`

In [ ]:
from IPython.display import Image as IPImage

def live(process, seconds=15, index=0, width=640):
    """Show the webcam video in the notebook for `seconds` seconds, changing each frame with process(photo) → photo. To stop, press I I."""
    cap = cv2.VideoCapture(index, cv2.CAP_DSHOW) if os.name == "nt" else cv2.VideoCapture(index)
    still = None if cap.isOpened() and cap.read()[0] else cv2.imread(PHOTO)   # no webcam → use the photo
    handle = display(None, display_id=True)
    t0, n, out = time.time(), 0, None
    try:
        while time.time() - t0 < seconds and (still is None or n < 10):
            if still is None:
                ok, f = cap.read()
                if not ok:
                    break
            else:
                f = still.copy()
            out = process(f)
            s = width / out.shape[1]
            small = cv2.resize(out, (width, int(out.shape[0] * s))) if s < 1 else out
            handle.update(IPImage(data=cv2.imencode(".jpg", small)[1].tobytes()))
            n += 1
    except KeyboardInterrupt:
        pass
    finally:
        cap.release()
    print(f"{n} frames · {n / max(time.time() - t0, 1e-6):.1f} FPS" + (" (no webcam, so simulated with the photo)" if still is not None else ""))
    return out

In [ ]:
from collections import Counter
stats = {"frames": 0, "with_face": 0, "front": 0}
moods = Counter()

def focus_meter(f):
    stats["frames"] += 1
    found = find_faces(f)
    msg, color = "no face", (0, 0, 255)
    if found:
        (x1, y1, x2, y2), _ = max(found, key=lambda it: (it[0][2] - it[0][0]) * (it[0][3] - it[0][1]))   # the biggest face
        info = analyze(f[y1:y2, x1:x2])
        stats["with_face"] += 1
        moods[info["emotion"]] += 1
        d = facing(info["yaw"], info["pitch"])
        if d == "front":
            stats["front"] += 1; msg, color = "focus", (0, 200, 0)
        else:
            msg, color = "looking away", (0, 160, 255)
        cv2.rectangle(f, (x1, y1), (x2, y2), color, 2)
    score = stats["front"] / stats["frames"] * 100
    cv2.putText(f, f"{msg}  score {score:.0f}%", (10, 34), cv2.FONT_HERSHEY_SIMPLEX, 1.0, color, 2)
    return f

live(focus_meter, seconds=20)
print(f"Focus score: {stats['front'] / max(stats['frames'], 1) * 100:.0f}%  (face found in {stats['with_face']}/{stats['frames']} frames)")
print("Expressions:", dict(moods.most_common()))

**Go further**
- Show a warning message when there is no face for more than 5 seconds
- Record the score every minute and draw a graph (`matplotlib`)
- Tune the threshold (`th` in `facing`) for your own seat